# Clasificador de lugares para estudiar con TensorC

Este notebook instala TensorC, permite subir el ZIP, lo descomprime y usa datos.csv y fotos/ desde la carpeta raíz.

In [ ]:
%pip install -q tensorc==0.14.0 numpy pillow pandas

from pathlib import Path
import sys
import csv
import zipfile
from google.colab import files
import json
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
import tensorc

print('Sube también el archivo train_study_classifier.py cuando Colab lo solicite.')
uploaded_script = files.upload()
EXPERIMENT_DIR = Path('/content')
sys.path.insert(0, str(EXPERIMENT_DIR))
from train_study_classifier import (
    StudyNetwork, split_rows, balanced_order, tensor_batch,
    one_hot, probabilities, metrics, write_predictions,
    metadata as encode_metadata,
)

print('TensorC:', tensorc.__version__)

In [ ]:
# Subir y descomprimir el ZIP del dataset.
uploaded_zip = files.upload()
zip_name = next(iter(uploaded_zip))
extract_dir = Path('/content/dataset')
extract_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(extract_dir)
csv_files = list(extract_dir.rglob('datos.csv'))
assert csv_files, 'No se encontró datos.csv dentro del ZIP'
DATASET_DIR = csv_files[0].parent
assert (DATASET_DIR / 'fotos').exists(), f'No existe fotos/: {DATASET_DIR}'
CSV_PATH = DATASET_DIR / 'datos.csv'
OUTPUT_DIR = Path('/content/resultados_tensorc')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
EPOCHS = 35
BATCH_SIZE = 16

print('Dataset:', DATASET_DIR)

In [ ]:
# Cargar CSV y fotografías desde la carpeta raíz.
with CSV_PATH.open(encoding='utf-8-sig', newline='') as f:
    rows = list(csv.DictReader(f))

splits, uncertain_rows = split_rows(rows, SEED)

def load_arrays_from_root(rows, unknown=False):
    images = []
    features = []
    for row in rows:
        image_path = DATASET_DIR / row['foto_local']
        with Image.open(image_path) as image:
            image = image.convert('RGB').resize((32, 32), Image.Resampling.LANCZOS)
            array = np.asarray(image, dtype=np.float32).transpose(2, 0, 1) / 255.0
        images.append(array)
        features.append(encode_metadata(row))
    images = np.ascontiguousarray(np.stack(images), dtype=np.float32)
    features = np.ascontiguousarray(np.stack(features), dtype=np.float32)
    labels = np.asarray(
        [255 if unknown else int(row['study_label'] == 'apto') for row in rows],
        dtype=np.int32,
    )
    return images, features, labels

data = {name: load_arrays_from_root(values) for name, values in splits.items()}
uncertain = load_arrays_from_root(uncertain_rows, unknown=True)

for name, values in splits.items():
    print(name, len(values), Counter(row['study_label'] for row in values))
print('inciertas con fotografía:', len(uncertain_rows))
print('forma de imágenes:', data['train'][0].shape)

In [ ]:
# Definir la red multimodal TensorC.
model = StudyNetwork()
print('Parámetros entrenables:', len(model.params))


In [ ]:
# Entrenamiento con muestreo balanceado.
best_loss = float('inf')
history = []
best_path = OUTPUT_DIR / 'study_classifier_notebook.tck'

for epoch in range(1, EPOCHS + 1):
    images, features, labels = data['train']
    order = balanced_order(labels, SEED + epoch)
    losses = []

    for start in range(0, len(order), BATCH_SIZE):
        indices = order[start:start + BATCH_SIZE]
        logits = model.forward(
            tensor_batch(images[indices]),
            tensor_batch(features[indices]),
        )
        loss = logits.softmax().cross_entropy(one_hot(labels[indices]))
        loss.backward()
        model.optimizer.step()
        losses.append(float(loss.data[0]))

    val_images, val_features, val_labels = data['val']
    val_prob = probabilities(model, val_images, val_features, BATCH_SIZE)
    val_loss = float(-np.mean(
        val_labels * np.log(np.maximum(val_prob, 1e-7)) +
        (1 - val_labels) * np.log(np.maximum(1 - val_prob, 1e-7))
    ))
    val_result = metrics(val_labels, val_prob)
    history.append({
        'epoch': epoch,
        'train_loss': float(np.mean(losses)),
        'val_loss': val_loss,
        **val_result,
    })

    print(
        f'Epoca {epoch:02d}/{EPOCHS} | '
        f'train_loss {np.mean(losses):.5f} | '
        f'val_loss {val_loss:.5f} | '
        f'val_acc {val_result["accuracy"]:.2%}'
    )

    if val_loss < best_loss:
        best_loss = val_loss
        model.save(best_path)

print('Mejor checkpoint:', best_path)

In [ ]:
# Evaluación final y predicciones.
model.load(best_path)

test_images, test_features, test_labels = data['test']
test_prob = probabilities(model, test_images, test_features, BATCH_SIZE)
test_result = metrics(test_labels, test_prob)

write_predictions(
    OUTPUT_DIR / 'test_predictions.csv',
    splits['test'],
    test_prob,
)

uncertain_prob = probabilities(model, uncertain[0], uncertain[1], BATCH_SIZE)
write_predictions(
    OUTPUT_DIR / 'uncertain_predictions.csv',
    uncertain_rows,
    uncertain_prob,
    unknown=True,
)

(OUTPUT_DIR / 'metrics.json').write_text(
    json.dumps(test_result, indent=2), encoding='utf-8'
)
(OUTPUT_DIR / 'history.json').write_text(
    json.dumps(history, indent=2), encoding='utf-8'
)

print('MÉTRICAS DE PRUEBA')
for name, value in test_result.items():
    print(f'{name}: {value}')

display(pd.read_csv(OUTPUT_DIR / 'test_predictions.csv').head())
display(pd.read_csv(OUTPUT_DIR / 'uncertain_predictions.csv').head())